# Private N21 Manual Face & Alignment QC — 24 paired images

**Private notebook: do not upload executed copy, images, source IDs or row-level results to public GitHub.** No model inference or generation. Displays original / N07 reference / N17 neutral-target images, then asks the reviewer to record **pass / uncertain / fail** for source-face visibility, output-face plausibility, correct pairing and obvious crop/alignment errors. These subjective judgments cannot be automated or marked PASS without human review.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json
import numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
R=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=R/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N17=R/'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
N18=R/'nb18_paired_neutral_pilot_analysis_v1'
OUT=R/'nb21_final_exact_n07_ensemble_fer'
rec=pd.read_csv(N17/'private_execution_receipts.csv')
ref=pd.read_csv(N18/'private_paired_reference_and_n17_manifest.csv')
assert len(rec)==24 and len(ref)==24
assert rec.source_id.nunique()==12 and not rec.duplicated(['source_id','seed']).any()
assert not ref.duplicated(['source_id','seed']).any()
assert set(rec.status)=={'VERIFIED'}
def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
 return h.hexdigest()
for x in rec.itertuples(index=False):
 assert Path(x.output_path).is_file() and sha(x.output_path)==x.output_sha256
rows=ref.merge(rec[['source_id','seed','output_path','output_sha256']],on=['source_id','seed'],how='inner',validate='one_to_one',suffixes=('_ref','_new'))
assert len(rows)==24
print('DATA INTEGRITY PASS — 24 source/seed pairs; no images exported.')
assert 'output_path' in ref.columns, 'N18 reference output_path not found'
print('Verified reference/output manifest paths.')


In [ ]:
REF_PATH_COL="output_path_ref"
NEW_PATH_COL="output_path_new"
assert REF_PATH_COL in rows.columns and NEW_PATH_COL in rows.columns
print("N07 reference and N17 output paths resolved")


In [ ]:
# Private contact-sheet review (8 batches x 3 pairs to remain readable).
# Do not download/share screenshots from this cell.
THUMB=(280,280)
def tile(path, text):
 im=Image.open(path).convert('RGB')
 im=ImageOps.contain(im,THUMB)
 canvas=Image.new('RGB',(300,325),'#eeeeee')
 canvas.paste(im,((300-im.width)//2,30+(280-im.height)//2))
 ImageDraw.Draw(canvas).text((8,8),text,fill='black')
 return canvas
rows=rows.sort_values(['source_id','seed']).reset_index(drop=True)
review_path=OUT/'private_manual_face_qc.csv'
if review_path.exists():
 review=pd.read_csv(review_path).fillna('')
else:
 review=rows[['source_id','seed']].copy()
 for c in ['pairing','original_face','n07_face','n17_face','alignment','notes','reviewer']:
  review[c]=''
for batch_start in range(0,len(rows),3):
 batch=rows.iloc[batch_start:batch_start+3]
 sheet=Image.new('RGB',(900,325*len(batch)),'white')
 for j,x in enumerate(batch.itertuples(index=False)):
  original=N07/'sources'/f'{x.source_id}.png'
  n07=Path(getattr(x,REF_PATH_COL))
  n17=Path(getattr(x,NEW_PATH_COL))
  if not all(p.is_file() for p in [original,n07,n17]):
   raise FileNotFoundError(f'Missing QC source/reference/output for {x.source_id} seed {x.seed}')
  for k,(path,label) in enumerate([(original,'ORIGINAL'),(n07,'N07 baseline'),(n17,'N17 neutral')]):
   sheet.paste(tile(path,f'{label} / pair {batch_start+j+1}'),(k*300,j*325))
 display(sheet)
 print(f'Review rows {batch_start+1} through {batch_start+len(batch)}. Mark PASS / UNCERTAIN / FAIL in the CSV, then continue.')
review.to_csv(review_path,index=False)
print('REVIEW TEMPLATE:',review_path)


## Human review and closure

After inspecting the pairs, open `private_manual_face_qc.csv` in Drive and populate each of **pairing**, **original_face**, **n07_face**, **n17_face**, **alignment** with `PASS`, `UNCERTAIN`, or `FAIL`; add brief notes/reviewer. Do not mark face recognition success based solely on a model-detected face. Save the file, then execute the next cell. If any item is uncertain or fails, keep the scientific evidence at `PENDING_MANUAL_QC` and report the affected count.


In [ ]:
review=pd.read_csv(review_path).fillna('')
assert len(review)==24 and not review.duplicated(['source_id','seed']).any()
fields=['pairing','original_face','n07_face','n17_face','alignment']
allowed={'PASS','UNCERTAIN','FAIL'}
counts={}
for field in fields:
 values=review[field].astype(str).str.strip().str.upper()
 counts[field]={v:int((values==v).sum()) for v in ['PASS','UNCERTAIN','FAIL','UNREVIEWED']}
 counts[field]['UNREVIEWED']=int((~values.isin(allowed)).sum())
all_pass=all(review[f].astype(str).str.strip().str.upper().eq('PASS').all() for f in fields)
report={'n_pairs':24,'manual_review_status':'PASS' if all_pass else 'PENDING_OR_HAS_FLAGS','counts':counts,'calibrated_reidentification_risk':False}
(OUT/'private_manual_qc_summary.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
print('Manual QC can only be completed by a reviewer actually inspecting the pairs.')
